In [1]:
# Move 10s images that overlap AIS ship times from train/ to test/
import os
import re
import shutil
from pathlib import Path
import pandas as pd

# Paths
ROOT = Path(__file__).parent if '__file__' in globals() else Path(os.getcwd())
AIS_CSV = ROOT.parent / 'AIS' / 'specific_ships.csv'
TRAIN_DIR = ROOT / 'train'
TEST_DIR = ROOT / 'test'

# Load AIS times
if not AIS_CSV.exists():
    raise FileNotFoundError(f"AIS CSV not found: {AIS_CSV}")

df = pd.read_csv(AIS_CSV)

# Column could be '__time_of_day' (our earlier code) or '_time_of_day' per request
col_candidates = ['__time_of_day', '_time_of_day', 'time_of_day']
col = next((c for c in col_candidates if c in df.columns), None)
if col is None:
    raise KeyError(f"None of time-of-day columns found in {AIS_CSV.name}: {col_candidates}")

# Parse HH:MM:SS into seconds-of-day
ship_seconds = []
for v in df[col].dropna().astype(str):
    m = re.match(r'^(\d{1,2}):(\d{2}):(\d{2})$', v.strip())
    if not m:
        continue
    h, mnt, s = map(int, m.groups())
    if 0 <= h < 24 and 0 <= mnt < 60 and 0 <= s < 60:
        ship_seconds.append(h * 3600 + mnt * 60 + s)

ship_seconds = sorted(set(ship_seconds))
print(f"Loaded {len(ship_seconds)} unique AIS times from {AIS_CSV.name}")

# Ensure directories
if not TRAIN_DIR.exists():
    raise FileNotFoundError(f"train directory not found: {TRAIN_DIR}")
TEST_DIR.mkdir(exist_ok=True)

# Helper to extract HHMMSS start from image filename (supports both 'HHMMSS_T0s.png' and stitched names)
# We take the first 6-digit token in the name.
re_hhmmss = re.compile(r'^(\d{6})')

moved = 0
skipped = 0

# Build list of train images
train_images = list(TRAIN_DIR.glob('*.png'))
print(f"Found {len(train_images)} images in {TRAIN_DIR}")

for img_path in train_images:
    name = img_path.stem  # without extension
    # Extract first 6 digits from the beginning; if not at start, try anywhere
    m = re_hhmmss.match(name)
    if not m:
        m = re.search(r'(\d{6})', name)
    if not m:
        skipped += 1
        continue
    hhmmss = m.group(1)
    try:
        h = int(hhmmss[0:2]); mnt = int(hhmmss[2:4]); sec = int(hhmmss[4:6])
    except Exception:
        skipped += 1
        continue
    start_sec = h * 3600 + mnt * 60 + sec
    end_sec = start_sec + 10  # exclusive

    # Does any AIS time fall within [start_sec, end_sec)?
    overlaps = any(start_sec <= s < end_sec for s in ship_seconds)
    if overlaps:
        dest = TEST_DIR / img_path.name
        try:
            shutil.move(str(img_path), str(dest))
            moved += 1
        except Exception as e:
            print(f"Failed to move {img_path.name}: {e}")

print(f"Moved {moved} images to {TEST_DIR}")
if skipped:
    print(f"Skipped {skipped} images without recognizable HHMMSS in filename")


Loaded 71 unique AIS times from specific_ships.csv
Found 341 images in c:\Users\45422\Desktop\DASProject\New_CAE\train
Moved 58 images to c:\Users\45422\Desktop\DASProject\New_CAE\test
